# boring — CUDA smoke test on Colab

Complements `tools/fake-nvcc` (which only checks that the host Rust code compiles, never real GPU execution): this notebook clones the repo, builds the `boring` compiler, transpiles a `.br` example to CUDA, then compiles **and runs** the generated project on a real NVIDIA GPU (Colab's T4).

**Before running**: `Runtime > Change runtime type > T4 GPU`, then push the branch you want to test (`git push origin <your-branch>`) — Colab clones from the remote, not from your local disk.

In [ ]:
# Allocated GPU + CUDA driver version
!nvidia-smi

In [ ]:
# CUDA toolkit (nvcc) is preinstalled on Colab — just check the version
!nvcc --version

In [ ]:
# Rust is NOT preinstalled on Colab, unlike CUDA
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --default-toolchain stable
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]
!rustc --version && cargo --version

## Clone the repo

`origin` (framagit) is authoritative; the GitHub mirror (`github.com/mlanoe/boring`) also works if framagit is down. Change `BRANCH` to the branch you want to test (push it first).

In [ ]:
BRANCH = "main"  # <- replace with your working branch
!git clone --branch $BRANCH --depth 1 https://framagit.org/maykeul/boring.git
%cd boring

In [ ]:
# Build the boring compiler itself (a Rust CLI, nothing GPU-specific here)
!cargo build --release --quiet
BORING = "./target/release/boring"
!$BORING --help | head -5

## Transpile an example to CUDA

`examples/vector_add_gpu.br` is used as the test case — swap in your own `.br` file if needed. Output goes to `examples/vector_add_gpu_cuda/` (naming convention: `<stem>_cuda`).

In [ ]:
SOURCE = "examples/vector_add_gpu.br"  # <- replace with the .br file to test
!$BORING build --target cuda $SOURCE

import pathlib
stem = pathlib.Path(SOURCE).stem
PROJECT_DIR = str(pathlib.Path(SOURCE).parent / f"{stem}_cuda")
print("Generated project:", PROJECT_DIR)
!ls $PROJECT_DIR

## `cudarc` feature matching the runtime's CUDA version

`cudarc` (0.19.x) dynamically loads `libcuda`/`libnvrtc`, but it still requires a concrete `cuda-XXXXX` feature at compile time (see `tools/fake-nvcc/README.md`). We derive it from `nvcc --version`; adjust manually if cudarc doesn't have that exact patch (list of supported features: https://docs.rs/crate/cudarc/latest/features).

In [ ]:
import re, subprocess

out = subprocess.run(["nvcc", "--version"], capture_output=True, text=True).stdout
m = re.search(r"release (\d+)\.(\d+)", out)
major, minor = m.group(1), m.group(2)
feature = f"cuda-{major}{int(minor):02d}0"
print(f"Detected CUDA {major}.{minor} -> candidate feature: {feature}")

cargo_toml = pathlib.Path(PROJECT_DIR) / "Cargo.toml"
text = cargo_toml.read_text()
text = re.sub(
    r'(cudarc\s*=\s*\{[^}]*features\s*=\s*\[)([^\]]*)(\])',
    lambda mo: f'{mo.group(1)}"driver", "nvrtc", "{feature}"{mo.group(3)}',
    text,
)
cargo_toml.write_text(text)
print(cargo_toml.read_text())

If the regex substitution above doesn't match (the `cudarc` dependency is written differently), open `Cargo.toml` in the Colab file browser and add the feature by hand.

In [ ]:
# Real build: nvcc compiles kernels/main.cu to PTX (build.rs), cargo compiles the host Rust code
!cd $PROJECT_DIR && cargo build --release --quiet

In [ ]:
# Run on the real T4 GPU -- this is the check fake-nvcc can NOT do
!cd $PROJECT_DIR && cargo run --release

## Notes

- A successful run here proves: the generated PTX compiles, `cudarc` loads the module, the kernel actually executes on a real NVIDIA GPU, and results come back correctly — exactly what `fake-nvcc` can't prove (it never touches the `.cu` file or runs anything).
- A Colab session is ephemeral: on every new session, rerun the cells in order (rustup, clone, build).
- To test unpushed changes: `git push origin <branch>` before rerunning the clone cell, or change `BRANCH`.